<a href="https://colab.research.google.com/github/2303A51803/Twitter_sentiment_Analysis/blob/main/Copy_of_NLP_Projec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [ ]:
data = pd.read_csv("twitter_training.csv", on_bad_lines='skip')
data.columns = ["id", "entity", "sentiment", "text"]

data = data[["text", "sentiment"]]
data = data[data["sentiment"].isin(["Positive", "Negative"])]
data["label"] = data["sentiment"].apply(lambda x: 1 if x == "Positive" else 0)

def clean_text(text):
    text = re.sub("[^a-zA-Z]", " ", str(text))
    text = text.lower()
    return text

data["clean_text"] = data["text"].apply(clean_text)

print(data.head())

                                                text sentiment  label  \
0  I am coming to the borders and I will kill you...  Positive      1   
1  im getting on borderlands and i will kill you ...  Positive      1   
2  im coming on borderlands and i will murder you...  Positive      1   
3  im getting on borderlands 2 and i will murder ...  Positive      1   
4  im getting into borderlands and i can murder y...  Positive      1   

                                          clean_text  
0  i am coming to the borders and i will kill you...  
1  im getting on borderlands and i will kill you ...  
2  im coming on borderlands and i will murder you...  
3  im getting on borderlands   and i will murder ...  
4  im getting into borderlands and i can murder y...  


Now that the data is loaded and cleaned, we can split it into training and testing sets and then vectorize the text data.

In [ ]:
X = data["clean_text"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)


In [ ]:

vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model_lr = LogisticRegression()
model_lr.fit(X_train_vec, y_train)

y_pred = model_lr.predict(X_test_vec)
print("✅ Logistic Regression Accuracy:", accuracy_score(y_test, y_pred))

✅ Logistic Regression Accuracy: 0.9070893371757925


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=100)
X_test_pad = pad_sequences(X_test_seq, maxlen=100)


model_lstm = Sequential()
model_lstm.add(Embedding(input_dim=5000, output_dim=32, input_length=100))
model_lstm.add(LSTM(64))
model_lstm.add(Dense(1, activation='sigmoid'))


model_lstm.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model_lstm.fit(X_train_pad, y_train, epochs=3, batch_size=32, validation_split=0.2, verbose=1)

loss, acc = model_lstm.evaluate(X_test_pad, y_test)
print("🤖 LSTM Model Accuracy:", acc)


Epoch 1/3


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


868/868 ━━━━━━━━━━━━━━━━━━━━ 32s 35ms/step - accuracy: 0.7165 - loss: 0.5574 - val_accuracy: 0.8644 - val_loss: 0.3299
Epoch 2/3
868/868 ━━━━━━━━━━━━━━━━━━━━ 30s 35ms/step - accuracy: 0.8869 - loss: 0.2724 - val_accuracy: 0.8879 - val_loss: 0.2692
Epoch 3/3
868/868 ━━━━━━━━━━━━━━━━━━━━ 30s 34ms/step - accuracy: 0.9118 - loss: 0.2086 - val_accuracy: 0.8938 - val_loss: 0.2600
272/272 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.8792 - loss: 0.2783
🤖 LSTM Model Accuracy: 0.8871469497680664


In [ ]:
print("\n📊 Model Comparison:")
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred))
print("LSTM Model Accuracy:", acc)



📊 Model Comparison:
Logistic Regression Accuracy: 0.9278005210271678
LSTM Model Accuracy: 0.9184964895248413


In [ ]:
def predict_tweet(text):
    text = re.sub("[^a-zA-Z]", " ", str(text))
    text = text.lower()

    # Logistic Regression Prediction
    text_vec = vectorizer.transform([text])
    pred_lr = model_lr.predict(text_vec)[0]

    # LSTM Prediction
    text_seq = tokenizer.texts_to_sequences([text])
    text_pad = pad_sequences(text_seq, maxlen=100)
    pred_lstm = model_lstm.predict(text_pad)[0][0]

    print("\n🔍 Tweet:", text)
    if pred_lr == 1:
        print("🟢 Logistic Regression says: Positive Tweet")
    else:
        print("🔴 Logistic Regression says: Negative Tweet")

    if pred_lstm > 0.5:
        print("🟢 LSTM says: Positive Tweet")
    else:
        print("🔴 LSTM says: Negative Tweet")

# Example tweet
predict_tweet("I love using this new phone, it’s awesome!")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 268ms/step

🔍 Tweet: i love using this new phone  it s awesome 
🟢 Logistic Regression says: Positive Tweet
🟢 LSTM says: Positive Tweet


In [ ]:
import sys
!{sys.executable} -m pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 27.0 MB/s eta 0:00:00


In [ ]:


import pandas as pd
import numpy as np
import re
import nltk
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
nltk.download('punkt')
nltk.download('punkt_tab')

# Load dataset

df = pd.read_csv("twitter_training.csv", header=None)   # change path if needed
df.columns = ["id", "entity", "sentiment", "text"]
df = df.dropna(subset=['text','sentiment'])

# Clean and tokenize

def clean_text(s):
    s = s.lower()
    s = re.sub(r'http\S+','', s)
    s = re.sub(r'@\w+','', s)
    s = re.sub(r'[^a-z0-9\s]',' ', s)
    s = re.sub(r'\s+',' ', s).strip()
    return s

df['text'] = df['text'].apply(clean_text)
df['tokens'] = df['text'].apply(nltk.word_tokenize)

# Train Word2Vec embeddings

w2v = Word2Vec(sentences=df['tokens'], vector_size=100, window=5, min_count=2, workers=4)

def avg_vector(tokens):
    vecs = [w2v.wv[t] for t in tokens if t in w2v.wv]
    if not vecs:
        return np.zeros(100)
    return np.mean(vecs, axis=0)

X = np.vstack(df['tokens'].apply(avg_vector))
y = df['sentiment'].apply(lambda x: 1 if x == "Positive" else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Accuracy: 0.7505405405405405

Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.94      0.84     10669
           1       0.63      0.26      0.37      4131

    accuracy                           0.75     14800
   macro avg       0.70      0.60      0.61     14800
weighted avg       0.73      0.75      0.71     14800



In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd # Import pandas to access 'data'
from sklearn.model_selection import train_test_split # Import train_test_split

# Re-create X, y, X_train, X_test, y_train, y_test for consistency with tokenizer and padding
# Assuming 'data' DataFrame and 'tokenizer' are still in the environment from previous cells
X_orig = data["clean_text"]
y_orig = data["label"]

X_train_bi, X_test_bi, y_train_bi, y_test_bi = train_test_split(X_orig, y_orig, test_size=0.2, random_state=0)

# The 'tokenizer' from cell cWZmpcX5gxQO is assumed to be in the environment
X_train_seq_bi = tokenizer.texts_to_sequences(X_train_bi)
X_test_seq_bi = tokenizer.texts_to_sequences(X_test_bi)

maxlen = 100 # Consistent with previous LSTM maxlen
X_train_pad_bi = pad_sequences(X_train_seq_bi, maxlen=maxlen)
X_test_pad_bi = pad_sequences(X_test_seq_bi, maxlen=maxlen)



num_words = len(tokenizer.word_index)

max_len = X_train_pad_bi.shape[1]
num_classes = 2

model = Sequential()
# Use the vocabulary size from Keras Tokenizer for input_dim
model.add(Embedding(input_dim=num_words, output_dim=100, input_length=max_len))
model.add(Bidirectional(LSTM(128)))
model.add(Dropout(0.3))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
# Use num_classes for the output layer and 'softmax' activation for multi-class (or 'sigmoid' for binary)
# Assuming binary classification based on previous cells
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

history = model.fit(X_train_pad_bi, y_train_bi, validation_split=0.1, epochs=8, batch_size=64, verbose=1)

y_pred_prob = model.predict(X_test_pad_bi)
y_pred = (y_pred_prob > 0.5).astype("int32")


print("Accuracy:", accuracy_score(y_test_bi, y_pred))
print("\nClassification report:\n", classification_report(y_test_bi, y_pred))

# model.save('bilstm_model.h5') # Use save for Keras models
# print("\n✅ Bi-LSTM model saved as bilstm_model.h5")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
488/488 ━━━━━━━━━━━━━━━━━━━━ 247s 496ms/step - accuracy: 0.7001 - loss: 0.5357 - val_accuracy: 0.8746 - val_loss: 0.2971
Epoch 2/8
488/488 ━━━━━━━━━━━━━━━━━━━━ 207s 424ms/step - accuracy: 0.8942 - loss: 0.2575 - val_accuracy: 0.9035 - val_loss: 0.2359
Epoch 3/8
488/488 ━━━━━━━━━━━━━━━━━━━━ 207s 425ms/step - accuracy: 0.9225 - loss: 0.1804 - val_accuracy: 0.9092 - val_loss: 0.2147
Epoch 4/8
488/488 ━━━━━━━━━━━━━━━━━━━━ 210s 431ms/step - accuracy: 0.9379 - loss: 0.1439 - val_accuracy: 0.9164 - val_loss: 0.2287
Epoch 5/8
488/488 ━━━━━━━━━━━━━━━━━━━━ 207s 424ms/step - accuracy: 0.9454 - loss: 0.1193 - val_accuracy: 0.9179 - val_loss: 0.2066
Epoch 6/8
488/488 ━━━━━━━━━━━━━━━━━━━━ 271s 444ms/step - accuracy: 0.9511 - loss: 0.1023 - val_accuracy: 0.9219 - val_loss: 0.2202
Epoch 7/8
488/488 ━━━━━━━━━━━━━━━━━━━━ 210s 431ms/step - accuracy: 0.9576 - loss: 0.0916 - val_accuracy: 0.9231 - val_loss: 0.2106
Epoch 8/8
488/488 ━━━━━━━━━━━━━━━━━━━━ 210s 431ms/step - accuracy: 0.9591 - loss: 0

In [ ]:
# -----------------------------
# LSTM + Attention Model
# -----------------------------
import tensorflow as tf
from tensorflow.keras.layers import Layer
from tensorflow.keras import backend as K

# Custom Attention Layer
class Attention(Layer):
    def __init__(self, **kwargs):
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight',
                                 shape=(input_shape[-1], 1),
                                 initializer='random_normal',
                                 trainable=True)
        self.b = self.add_weight(name='attention_bias',
                                 shape=(input_shape[1], 1),
                                 initializer='zeros',
                                 trainable=True)
        super(Attention, self).build(input_shape)

    def call(self, inputs):
        e = K.tanh(K.dot(inputs, self.W) + self.b)
        e = K.squeeze(e, axis=-1)
        alpha = K.softmax(e)
        alpha = K.expand_dims(alpha, axis=-1)
        context = inputs * alpha
        return K.sum(context, axis=1)

# Build model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model_attention = Sequential()
model_attention.add(Embedding(input_dim=5000, output_dim=32, input_length=100))
model_attention.add(LSTM(64, return_sequences=True))
model_attention.add(Attention())
model_attention.add(Dense(1, activation='sigmoid'))

model_attention.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model_attention.summary()

# Train using the consistent data from the Bi-LSTM setup
model_attention.fit(X_train_pad_bi, y_train_bi, epochs=3, batch_size=32, validation_split=0.2, verbose=1)

# Evaluate using the consistent data from the Bi-LSTM setup
loss_att, acc_att = model_attention.evaluate(X_test_pad_bi, y_test_bi)
print("🌟 LSTM + Attention Accuracy:", acc_att)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention (Attention)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
868/868 ━━━━━━━━━━━━━━━━━━━━ 60s 65ms/step - accuracy: 0.6341 - loss: 0.6053 - val_accuracy: 0.8340 - val_loss: 0.3734
Epoch 2/3
868/868 ━━━━━━━━━━━━━━━━━━━━ 56s 64ms/step - accuracy: 0.8641 - loss: 0.3219 - val_accuracy: 0.8638 - val_loss: 0.3207
Epoch 3/3
868/868 ━━━━━━━━━━━━━━━━━━━━ 82s 64ms/step - accuracy: 0.8963 - loss: 0.2459 - val_accuracy: 0.8886 - val_loss: 0.2695
272/272 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - accuracy: 0.8811 - loss: 0.2922
🌟 LSTM + Attention Accuracy: 0.8805763721466064


In [ ]:
!pip install transformers


In [ ]:
import tensorflow as tf
from transformers import BertTokenizer, TFBertModel
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import os
import shutil

# Clear Hugging Face cache to remove any corrupted or incompatible model files
cache_dir = os.path.expanduser('~/.cache/huggingface')
if os.path.exists(cache_dir):
    print(f"Clearing Hugging Face cache at {cache_dir}")
    shutil.rmtree(cache_dir)
else:
    print(f"Hugging Face cache directory not found at {cache_dir}")

# Force install known stable versions of transformers
!pip install transformers==4.41.2 --upgrade --quiet

# --------------------------------------------------------
# 1️⃣ Load dataset and clean
# --------------------------------------------------------
data = pd.read_csv("twitter_training.csv", on_bad_lines='skip')
data.columns = ["id","entity","sentiment","text"]

data = data[data["sentiment"].isin(["Positive", "Negative"])]
data["label"] = data["sentiment"].apply(lambda x: 1 if x=="Positive" else 0)

def clean_text(t):
    t = re.sub("[^a-zA-Z ]", " ", str(t))
    return t.lower().strip()

data["clean"] = data["text"].apply(clean_text)

# ✅ Reduce dataset size so steps-per-epoch = 500
# steps = samples / batch_size
# samples = 500 * 16 = 8000
data = data.sample(8000, random_state=42)

X = data["clean"].tolist()
y = np.array(data["label"])


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --------------------------------------------------------
# 2️⃣ Tokenizer
# --------------------------------------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def encode(text_list):
    return tokenizer(
        text_list,
        padding=True,
        truncation=True,
        max_length=100, # Explicitly set max_length to match model input
        return_tensors="tf"
    )

train_enc = encode(X_train)
test_enc  = encode(X_test)

# --------------------------------------------------------
# 3️⃣ Load BERT base (NO classification head)
# --------------------------------------------------------
# Explicitly set use_safetensors=False to avoid previous TypeErrors related to loading.
bert_model_raw = TFBertModel.from_pretrained("bert-base-uncased", force_download=True, use_safetensors=False)

# ✅ Freeze BERT layers to speed up training (VERY IMPORTANT)
for layer in bert_model_raw.layers:
    layer.trainable = False


# Define a custom Keras Layer to wrap the TFBertModel
class BertLayer(tf.keras.layers.Layer):
    def __init__(self, bert_model, **kwargs):
        super(BertLayer, self).__init__(**kwargs)
        self.bert = bert_model

    def call(self, inputs):
        # inputs should be a dictionary with 'input_ids' and 'attention_mask'
        input_ids = inputs['input_ids']
        attention_mask = inputs['attention_mask']
        bert_output = self.bert(input_ids, attention_mask=attention_mask)
        return bert_output.last_hidden_state

    def compute_output_shape(self, input_shape):
        # The output of bert_output.last_hidden_state is (batch_size, sequence_length, hidden_size)
        # For 'bert-base-uncased', hidden_size is 768. input_shape is a dict of shapes for input_ids and attention_mask
        # input_shape['input_ids'][0] is batch_size, input_shape['input_ids'][1] is sequence_length (max_len)
        return (input_shape['input_ids'][0], input_shape['input_ids'][1], 768)

    def get_config(self):
        config = super(BertLayer, self).get_config()
        return config

# Wrap the TFBertModel in the custom Keras Layer
bert_layer = BertLayer(bert_model_raw)

# 4️⃣ Build BERT + LSTM model
# max_len is not defined, assuming it should be 100 based on tokenizer max_length
max_len = 100
input_ids = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name="attention_mask")

# Pass inputs to the custom BertLayer
sequence_output = bert_layer({'input_ids': input_ids, 'attention_mask': attention_mask})

lstm_out = tf.keras.layers.LSTM(128)(sequence_output)
drop = tf.keras.layers.Dropout(0.3)(lstm_out)
final = tf.keras.layers.Dense(1, activation="sigmoid")(drop)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=final)

model.compile(
    optimizer=tf.keras.optimizers.Adam(2e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# --------------------------------------------------------
# 5️⃣ Train
# --------------------------------------------------------
history = model.fit(
    [train_enc["input_ids"], train_enc["attention_mask"]],
    y_train,
    validation_split=0.1,
    epochs=2,
    batch_size=16
)


loss, acc = model.evaluate(
    [test_enc["input_ids"], test_enc["attention_mask"]],
    y_test
)
print(f"\n🔥 BERT + LSTM Accuracy: {acc:.4f}")

# Prediction
pred_probs = model.predict([test_enc["input_ids"], test_enc["attention_mask"]])
pred_labels = (pred_probs > 0.5).astype(int)

print("\n📊 Classification Report:")
print(classification_report(y_test, pred_labels))

# --------------------------------------------------------
# 7️⃣ Save
# --------------------------------------------------------
model.save("bert_lstm_final_model")
print("\n✅ Model Saved")

Clearing Hugging Face cache at /root/.cache/huggingface


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tf_model.h5:   0%|          | 0.00/536M [00:00<?, ?B/s]

Some layers from the model checkpoint at bert-base-uncased were not used when initializing TFBertModel: ['nsp___cls', 'mlm___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertModel were initialized from the model checkpoint at bert-base-uncased.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions without further training.


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 attention_mask (InputLayer  [(None, 100)]                0         []                            
 )                                                                                                
                                                                                                  
 input_ids (InputLayer)      [(None, 100)]                0         []                            
                                                                                                  
 bert_layer (BertLayer)      (None, 100, 768)             1094822   ['attention_mask[0][0]',      
                                                          40         'input_ids[0][0]']           
                                                                                              

ValueError: in user code:

    File "/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py", line 2436, in predict_function  *
        return step_function(self, iterator)
    File "/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py", line 2421, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py", line 2409, in run_step  **
        outputs = model.predict_step(data)
    File "/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py", line 2377, in predict_step
        return self(x, training=False)
    File "/usr/local/lib/python3.12/dist-packages/tf_keras/src/utils/traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File "/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/input_spec.py", line 298, in assert_input_compatibility
        raise ValueErr